# LoRA Finetuning Notebook

Finetunes a LoRA adapter on StableDiffusion 1.5's UNet.


Runtime: Colab A100


Datset: Naruto BLIP captions


Pipeline: clone repo --> install dependencies/download pretrained weights --> mount Drive --> write training config --> train. 


Outputs: checkpoints + `losses.json` + snapshot of training config are saved to `runs/<run_name>/`, symlinked to Google Drive.

## Environment
Confirm a GPU is available (should print `cuda`)

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## Repo and dependencies
Clone the repo and install `sdrebuilt` as a package. `--no-deps` is used to avoid overwriting Colab's preinstalled `torch`/`numpy` with the version the `sdrebuilt` package uses.

In [3]:
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

Cloning into 'stable-diffusion-rebuilt'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (358/358), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 358 (delta 180), reused 293 (delta 115), pack-reused 0 (from 0)
Receiving objects: 100% (358/358), 3.67 MiB | 36.50 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/stable-diffusion-rebuilt
Already up to date.


In [4]:
# install dependencies (skip torch/numpy)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


## Pretrained weights
Download the SD1.5 checkpoint from HuggingFace. Should be ~4.0 GB

In [5]:
# download pretrained weights from HF
!python scripts/download_data.py
!ls -lh data/weights/   # should be ~4.0 GB

downloading data/weights/v1-5-pruned-emaonly.safetensors
total 4.0G
-rw-r--r-- 1 root root 4.0G Jul 13 07:08 v1-5-pruned-emaonly.safetensors


## Mount drive and create symlink
runs/ is gitignored and Colab's disk is wiped when the kernel disconnects, so `runs/` is symlinked to a Google Drive folder, which training writes to.

In [6]:
# mount drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
# add symlink from colab runs/ -> content/drive/MyDrive/sd-rebuilt/runs/
!ln -s /content/drive/MyDrive/sd-rebuilt/runs/ runs
!ls -la runs/

total 4
drwx------ 2 root root 4096 Jul 10 22:40 naruto_r16_selfcross_lora_test


## Training run config
- rank: 16
- alpha: 16
- targets: self and cross attention q/k/v/out projections
- batch size: 8
- learning rate: 1e-4
- n_epochs: 10

Written to `configs/lora.yaml`. The `runs/<run_name>` for this run comes from this config.

In [8]:
%%writefile configs/lora.yaml
# training run 1
name: lora
pretrained_path: data/weights/v1-5-pruned-emaonly.safetensors
device: cuda

dataset:
  type: HuggingFace
  name: naruto

r: 16
alpha: 16
targets:
  desc: selfcross
  layers: ["q1", "k1", "v1", "proj_out1", "q2", "k2", "v2", "proj_out2"]

n_epochs: 10
batch_size: 8
lr: 0.0001
seed: 42
log_interval: 10

Overwriting configs/lora.yaml


## Train
Precomputes and caches image/caption embeddings on first run of the dataset to `data/cache/naruto/`, then trains LoRA-injected UNet. Checkpoints save per epoch.

In [9]:
!python scripts/train_lora.py


>>> Loading UNet and injecting LoRA layers

>>> Preparing dataset

>>> Precomputing image/caption embeddings (no cache found)
tokenizer_config.json: 100% 905/905 [00:00<00:00, 5.47MB/s]
vocab.json: 100% 961k/961k [00:00<00:00, 106MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 103MB/s]
tokenizer.json: 100% 2.22M/2.22M [00:00<00:00, 129MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 2.97MB/s]
README.md: 100% 1.02k/1.02k [00:00<00:00, 2.76MB/s]
Repo card metadata block was not found. Setting CardData to empty.
dataset_infos.json: 100% 897/897 [00:00<00:00, 4.48MB/s]
data/train-00000-of-00002-12944970063701(…): 100% 344M/344M [00:04<00:00, 78.2MB/s]
data/train-00001-of-00002-cefa2f480689f1(…): 100% 357M/357M [00:04<00:00, 73.5MB/s]
Encoding image/caption batches: 100% 153/153 [01:47<00:00,  1.42it/s]

>>> Training
epoch 1/10: 100% 153/153 [00:32<00:00,  4.68it/s]
epoch 2/10: 100% 153/153 [00:31<00:00,  4.82it/s]
epoch 3/10: 100% 153/153 [00:32<00:00,  4.78it/s]
epoch 4/10: 100% 

## Outputs
- Checkpoints: `runs/<run_name>/checkpoints/checkpoint-<epoch>.pt`
- Loss curve: `runs/<run_name>/losses.json`
- Frozen config: `runs/<run_name>/training_config.yaml`

Generate samples from a checkpoint with `scripts/generate.py`.